<a href="https://colab.research.google.com/github/K-vino/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K-vino/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

## 1. Unit of analysis + time window


- One row represents the daily performance of one content page for one client on one report date.
- I am using the `fact_content_daily_performance_sample.parquet` table.
- This sample contains the final month (June 2026).
- My goal is to support ranking or prioritizing content for refresh using historical performance signals.
- I deliberately exclude any future or label-derived information to avoid data leakage.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [27]:
print("Dataset shape:", df.shape)
print("Date range:")
print(df["report_date"].min())
print(df["report_date"].max())

print("\nUnique months:")
print(df["month"].unique())

Dataset shape: (11694072, 31)
Date range:
2026-06-01
2026-06-30

Unique months:
['2026-06']


In [28]:
!pip -q install datasets huggingface_hub duckdb pyarrow

## 2. Fields: feature / label / context / excluded

## Feature
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

## Label / Proxy
- Content refresh priority based on observed historical performance.

## Context
- report_date
- client_hash_id
- content_hash_id
- month

## Excluded
- Any future outcome or label-derived field.
- Client identifiers are used only for grouping and not as predictive features.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Verification query code that supports the claims made in the Markdown.*

The assignment asks for three verification queries.

Query 1 — Grain

In [30]:
print(df[
    ["report_date","client_hash_id","content_hash_id"]
].head())

  report_date           client_hash_id           content_hash_id
0  2026-06-01  client_3ffa76342f366962  content_1a6296faee432dae
1  2026-06-01  client_3ffa76342f366962  content_73f21e612565035a
2  2026-06-01  client_3ffa76342f366962  content_5a5be514ff559598
3  2026-06-01  client_3ffa76342f366962  content_05b377d0c8a5cfd8
4  2026-06-01  client_3ffa76342f366962  content_dc34c661d63e55a9


Query 2 — Row count and date span

In [31]:
print("Rows:", len(df))
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

Rows: 11694072
Start: 2026-06-01
End: 2026-06-30


Query 3 — Availability

In [32]:
available = df[
    (df["client_has_gsc"] == True) &
    (df["gsc_data_available"] == True)
]

print("Rows with GSC available:", len(available))

Rows with GSC available: 3878937


Five-feature frame

In [33]:
features = df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
]

features.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,0,0,NaN,0.0,0.0
1,0,0,NaN,0.0,0.0
2,0,0,NaN,0.0,0.0
3,0,0,NaN,0.0,0.0
4,0,0,NaN,0.0,0.0


- gsc_impressions — available before the refresh decision.
- gsc_clicks — historical Search Console metric.
- gsc_avg_position — historical search ranking.
- ga4_sessions — historical website traffic.
- scroll_events — historical engagement metric.

Leakage experiment

The assignment requires you to show a leaking feature and then remove it.

In [34]:
leak_features = features.copy()

leak_features["leak"] = df["gsc_clicks"]

print(leak_features.head())

leak_features = leak_features.drop(columns=["leak"])

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  scroll_events  \
0                0           0               NaN           0.0            0.0   
1                0           0               NaN           0.0            0.0   
2                0           0               NaN           0.0            0.0   
3                0           0               NaN           0.0            0.0   
4                0           0               NaN           0.0            0.0   

   leak  
0     0  
1     0  
2     0  
3     0  
4     0  


Adding a label-derived feature can make evaluation appear unrealistically good because it contains outcome information. After demonstrating the issue, the leaking feature was removed.

## 4. Data limits


This sample contains only the final month (June 2026), so it should be used for testing query mechanics rather than developing future labels. The data is observational and may not capture all factors affecting content performance. Some metrics may also be unavailable for rows where GSC or GA4 data is not available.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.